# Experimento 03 — Onboarding de Clientes Multi-Agent LLM
**PROJENER.AI SL · UAX · Máster Big Data · 2026**

Autora: Edurne Martínez de Contrasta  
Empresa: ACME Industrial S.A.  
Framework: CrewAI · Groq API · Python 3.12  

**Objetivo:** Replicar el framework multi-agente en el proceso de onboarding de clientes para validar la generalización de los hallazgos de los Experimentos 01 y 02.

**Estructura:**
- 20 casos con ground truth anotado: Simple (C01-C08), Medio (C09-C14), Complejo (C15-C20)
- 6 casos requieren escalación HiL — todos de nivel Complejo
- 4 agentes: Comercial, Legal, Finanzas, Compliance (decisor)
- Modelos: M1 (RPA baseline), M2 (Single Agent 8b), M4 (Multi-Agent mixto 8b×3+70b×1)
- Métricas: ARR, HDA, DER, PT

In [1]:
import sys, re, time, json, os
os.environ["GROQ_API_KEY"] = "TOKEN GROQ"
os.environ["OTEL_SDK_DISABLED"] = "true"

print(f"Python: {sys.version[:6]}")
print(f"Dataset: {os.path.exists('casos_onboarding.json')}")

Python: 3.12.1
Dataset: True


## APIs Mock — Onboarding de Clientes

In [2]:
def verificar_cliente(cliente_id, nombre, pais, tipo_cliente):
    """API mock — verificación comercial básica del cliente"""
    paises_riesgo_alto = ["Irán", "Corea del Norte", "Siria", "Rusia", "Bielorrusia"]
    paises_riesgo_medio = ["Emiratos Árabes Unidos", "China", "Turquía", "México"]
    
    riesgo_pais = "alto" if pais in paises_riesgo_alto else \
                  "medio" if pais in paises_riesgo_medio else "bajo"
    
    return {
        "cliente_id": cliente_id,
        "nombre": nombre,
        "tipo": tipo_cliente,
        "pais": pais,
        "riesgo_pais": riesgo_pais,
        "requiere_due_diligence": riesgo_pais in ["alto", "medio"],
        "estado": "pendiente_validacion"
    }

def verificar_compliance_kyc(caso):
    """API mock — KYC/AML/screening de sanciones"""
    alerta_aml = caso.get("alerta_aml", False)
    alerta_ubo = caso.get("alerta_beneficial_owner", False)
    pep = caso.get("pep", False)
    uso_dual = caso.get("uso_dual", False)
    alerta_pais = caso.get("alerta_pais", False)
    estructura_opaca = caso.get("estructura_opaca", False)
    conflicto = caso.get("conflicto_interes", False)
    
    nivel = "critico" if (alerta_aml or alerta_ubo or pep) else \
            "alto" if (uso_dual or alerta_pais or conflicto) else \
            "medio" if estructura_opaca else "bajo"
    
    return {
        "alerta_aml": alerta_aml,
        "alerta_ubo_sancionado": alerta_ubo,
        "pep_detectado": pep,
        "uso_dual": uso_dual,
        "alerta_pais_riesgo": alerta_pais,
        "estructura_opaca": estructura_opaca,
        "conflicto_interes": conflicto,
        "nivel_riesgo_compliance": nivel,
        "requiere_hil": nivel in ["critico", "alto"],
        "accion_recomendada": "bloquear_escalar" if nivel == "critico" else \
                              "due_diligence_reforzada" if nivel == "alto" else \
                              "documentacion_adicional" if nivel == "medio" else "aprobar"
    }

def verificar_credito(volumen_anual_eur, antiguedad_años, documentacion_completa, tipo_cliente):
    """API mock — análisis de riesgo financiero y límite de crédito"""
    if not documentacion_completa:
        riesgo = "alto"
        limite = 0
    elif antiguedad_años < 1:
        riesgo = "alto"
        limite = min(volumen_anual_eur * 0.1, 5000)
    elif antiguedad_años < 3:
        riesgo = "medio"
        limite = min(volumen_anual_eur * 0.3, 30000)
    elif volumen_anual_eur > 200000:
        riesgo = "medio"
        limite = min(volumen_anual_eur * 0.5, 150000)
    else:
        riesgo = "bajo"
        limite = min(volumen_anual_eur * 0.8, 100000)
    
    return {
        "volumen_solicitado": volumen_anual_eur,
        "limite_credito_aprobado": limite,
        "riesgo_financiero": riesgo,
        "documentacion_completa": documentacion_completa,
        "requiere_aval": riesgo == "alto" and tipo_cliente == "B2B",
        "aprobado": riesgo != "alto" or documentacion_completa
    }

def verificar_legal_contrato(pais, sector, volumen_anual_eur, uso_dual=False):
    """API mock — validación legal y tipo de contrato requerido"""
    contrato_internacional = pais not in ["España"]
    requiere_nda = volumen_anual_eur > 50000 or sector in ["tecnologia", "defensa_tecnologia"]
    requiere_licencia_exportacion = uso_dual
    
    return {
        "pais": pais,
        "contrato_internacional": contrato_internacional,
        "requiere_nda": requiere_nda,
        "requiere_licencia_exportacion": requiere_licencia_exportacion,
        "tipo_contrato": "internacional" if contrato_internacional else "estandar",
        "riesgo_legal": "alto" if requiere_licencia_exportacion else \
                        "medio" if contrato_internacional else "bajo"
    }

def registrar_cliente(caso):
    """API mock — registro del cliente en el sistema"""
    return {
        "cliente_id": f"CLI-2026-{caso['id']}",
        "estado": "pendiente_aprobacion",
        "tipo": caso.get("tipo_cliente", "B2B"),
        "nombre": caso.get("nombre", ""),
        "volumen_anual": caso.get("volumen_anual_eur", 0)
    }

## Dataset — 20 Casos de Onboarding

In [3]:
with open("casos_onboarding.json", encoding="utf-8") as f:
    data = json.load(f)
casos_onb = data["casos"]

print(f"✅ {len(casos_onb)} casos cargados")
print(f"   Simple:   {sum(1 for c in casos_onb if c['nivel']=='Simple')}")
print(f"   Medio:    {sum(1 for c in casos_onb if c['nivel']=='Medio')}")
print(f"   Complejo: {sum(1 for c in casos_onb if c['nivel']=='Complejo')}")
print(f"   Requieren HiL: {sum(1 for c in casos_onb if c['ground_truth']['requiere_hil'])}")
print(f"   B2B: {sum(1 for c in casos_onb if c['tipo_cliente']=='B2B')}")
print(f"   B2C: {sum(1 for c in casos_onb if c['tipo_cliente']=='B2C')}")

✅ 20 casos cargados
   Simple:   8
   Medio:    6
   Complejo: 6
   Requieren HiL: 6
   B2B: 15
   B2C: 5


In [4]:
def calcular_metricas(resultados, total):
    resueltos = sum(1 for r in resultados
                    if not r["escalo_hil"] and r["decision"] != "error")
    hil_req   = [r for r in resultados if r["ground_truth"].get("requiere_hil")]
    hil_det   = [r for r in hil_req    if r["escalo_hil"]]
    errores   = [r for r in resultados
                 if r["decision"] != r["ground_truth"].get("resultado","")
                 and not r["escalo_hil"] and r["decision"] != "error"]
    ARR = round(resueltos / total * 100, 1)
    HDA = round(len(hil_det) / len(hil_req) * 100, 1) if hil_req else 0
    DER = round(len(errores) / total * 100, 1)
    PT  = round(sum(r["tiempo"] for r in resultados) / total, 2)
    return ARR, HDA, DER, PT

os.makedirs("resultados_onboarding", exist_ok=True)
print("✅ Función métricas lista")

✅ Función métricas lista


## M1 — Baseline RPA

In [5]:
def procesar_onboarding_m1(caso):
    """Baseline RPA — reglas if-then para onboarding"""
    import time
    t_ini = time.time()

    alerta_aml   = caso.get("alerta_aml", False)
    alerta_ubo   = caso.get("alerta_beneficial_owner", False)
    pep          = caso.get("pep", False)
    uso_dual     = caso.get("uso_dual", False)
    alerta_pais  = caso.get("alerta_pais", False)
    conflicto    = caso.get("conflicto_interes", False)
    doc_completa = caso.get("documentacion_completa", True)
    volumen      = caso.get("volumen_anual_eur", 0)

    if alerta_aml or alerta_ubo or pep:
        decision = "bloqueo_escalacion_critica"
        escala   = False  # RPA no escala al humano
    elif uso_dual or alerta_pais or conflicto:
        decision = "rechazo_riesgo_alto"
        escala   = False
    elif not doc_completa:
        decision = "solicitud_documentacion"
        escala   = False
    elif volumen > 200000:
        decision = "aprobacion_con_limite"
        escala   = False
    else:
        decision = "aprobacion_automatica"
        escala   = False

    t_fin = time.time()
    return {
        "caso_id": caso["id"], "nivel": caso["nivel"],
        "decision": decision, "escalo_hil": escala,
        "tiempo": round(t_fin - t_ini, 4),
        "ground_truth": caso["ground_truth"]
    }

In [6]:
print("Ejecutando M1 — Baseline RPA (20 casos)...\n")
resultados_onb_m1 = []
for caso in casos_onb:
    r = procesar_onboarding_m1(caso)
    resultados_onb_m1.append(r)
    gt = r["ground_truth"]["resultado"]
    ok = "✅" if r["decision"] == gt else "❌"
    print(f"  {caso['id']} [{caso['nivel']}]: {r['decision']} | GT: {gt} {ok}")

ARR,HDA,DER,PT = calcular_metricas(resultados_onb_m1, 20)
print(f"\n{'='*40}")
print(f"M1 — ARR={ARR}% | HDA={HDA}% | DER={DER}% | PT={PT}s")
print(f"{'='*40}")
with open("resultados_onboarding/resultados_onb_m1.json","w",encoding="utf-8") as f:
    json.dump({"modelo":"M1_RPA_onboarding",
               "metricas":{"ARR":ARR,"HDA":HDA,"DER":DER,"PT":PT},
               "resultados_por_caso":resultados_onb_m1}, f, ensure_ascii=False, indent=2)
print("✅ Guardado resultados_onb_m1.json")

Ejecutando M1 — Baseline RPA (20 casos)...

  C01 [Simple]: aprobacion_automatica | GT: aprobacion_automatica ✅
  C02 [Simple]: aprobacion_automatica | GT: aprobacion_automatica ✅
  C03 [Simple]: aprobacion_automatica | GT: aprobacion_automatica ✅
  C04 [Simple]: aprobacion_automatica | GT: aprobacion_automatica ✅
  C05 [Simple]: aprobacion_automatica | GT: aprobacion_con_limite_credito ❌
  C06 [Simple]: aprobacion_automatica | GT: aprobacion_automatica ✅
  C07 [Simple]: aprobacion_automatica | GT: aprobacion_automatica ✅
  C08 [Simple]: aprobacion_automatica | GT: aprobacion_con_revision_anual ❌
  C09 [Medio]: aprobacion_automatica | GT: aprobacion_con_contrato_bilingue ❌
  C10 [Medio]: solicitud_documentacion | GT: solicitud_documentacion_adicional ❌
  C11 [Medio]: aprobacion_automatica | GT: aprobacion_con_justificacion_uso ❌
  C12 [Medio]: aprobacion_automatica | GT: aprobacion_con_garantia_pago ❌
  C13 [Medio]: aprobacion_automatica | GT: aprobacion_con_analisis_grupo ❌
  C14 [Med

M1 resuelve todo mecánicamente (ARR=100%) pero no detecta ningún caso HiL (HDA=0%) y comete errores en casos complejos (DER=70%). El RPA no distingue entre "aprobar" y "aprobar con condiciones" — aprueba todo lo que no tiene alerta explícita.

## M2 — Single Agent LLM (llama-3.1-8b-instant)

In [8]:
import json, sys, re, time, os
from crewai import Agent, Task, Crew, LLM

# Cargar dataset
with open("casos_onboarding.json", encoding="utf-8") as f:
    data = json.load(f)
casos_onb = data["casos"]

def calcular_metricas(resultados, total):
    resueltos = sum(1 for r in resultados if not r["escalo_hil"] and r["decision"] != "error")
    hil_req = [r for r in resultados if r["ground_truth"].get("requiere_hil")]
    hil_det = [r for r in hil_req if r["escalo_hil"]]
    errores = [r for r in resultados if r["decision"] != r["ground_truth"].get("resultado","") and not r["escalo_hil"] and r["decision"] != "error"]
    ARR = round(resueltos / total * 100, 1)
    HDA = round(len(hil_det) / len(hil_req) * 100, 1) if hil_req else 0
    DER = round(len(errores) / total * 100, 1)
    PT  = round(sum(r["tiempo"] for r in resultados) / total, 2)
    return ARR, HDA, DER, PT

def verificar_cliente(cliente_id, nombre, pais, tipo_cliente):
    paises_riesgo_alto = ["Irán", "Corea del Norte", "Siria", "Rusia", "Bielorrusia"]
    paises_riesgo_medio = ["Emiratos Árabes Unidos", "China", "Turquía", "México"]
    riesgo_pais = "alto" if pais in paises_riesgo_alto else "medio" if pais in paises_riesgo_medio else "bajo"
    return {"cliente_id": cliente_id, "nombre": nombre, "tipo": tipo_cliente, "pais": pais,
            "riesgo_pais": riesgo_pais, "requiere_due_diligence": riesgo_pais in ["alto","medio"], "estado": "pendiente_validacion"}

def verificar_compliance_kyc(caso):
    alerta_aml = caso.get("alerta_aml", False)
    alerta_ubo = caso.get("alerta_beneficial_owner", False)
    pep = caso.get("pep", False)
    uso_dual = caso.get("uso_dual", False)
    alerta_pais = caso.get("alerta_pais", False)
    estructura_opaca = caso.get("estructura_opaca", False)
    conflicto = caso.get("conflicto_interes", False)
    nivel = "critico" if (alerta_aml or alerta_ubo or pep) else "alto" if (uso_dual or alerta_pais or conflicto) else "medio" if estructura_opaca else "bajo"
    return {"alerta_aml": alerta_aml, "alerta_ubo_sancionado": alerta_ubo, "pep_detectado": pep,
            "uso_dual": uso_dual, "alerta_pais_riesgo": alerta_pais, "estructura_opaca": estructura_opaca,
            "conflicto_interes": conflicto, "nivel_riesgo_compliance": nivel,
            "requiere_hil": nivel in ["critico","alto"],
            "accion_recomendada": "bloquear_escalar" if nivel=="critico" else "due_diligence_reforzada" if nivel=="alto" else "documentacion_adicional" if nivel=="medio" else "aprobar"}

def verificar_credito(volumen_anual_eur, antiguedad_años, documentacion_completa, tipo_cliente):
    if not documentacion_completa: riesgo="alto"; limite=0
    elif antiguedad_años < 1: riesgo="alto"; limite=min(volumen_anual_eur*0.1,5000)
    elif antiguedad_años < 3: riesgo="medio"; limite=min(volumen_anual_eur*0.3,30000)
    elif volumen_anual_eur > 200000: riesgo="medio"; limite=min(volumen_anual_eur*0.5,150000)
    else: riesgo="bajo"; limite=min(volumen_anual_eur*0.8,100000)
    return {"volumen_solicitado": volumen_anual_eur, "limite_credito_aprobado": limite,
            "riesgo_financiero": riesgo, "documentacion_completa": documentacion_completa,
            "requiere_aval": riesgo=="alto" and tipo_cliente=="B2B", "aprobado": riesgo!="alto" or documentacion_completa}

def verificar_legal_contrato(pais, sector, volumen_anual_eur, uso_dual=False):
    contrato_internacional = pais not in ["España"]
    requiere_nda = volumen_anual_eur > 50000 or sector in ["tecnologia","defensa_tecnologia"]
    return {"pais": pais, "contrato_internacional": contrato_internacional,
            "requiere_nda": requiere_nda, "requiere_licencia_exportacion": uso_dual,
            "tipo_contrato": "internacional" if contrato_internacional else "estandar",
            "riesgo_legal": "alto" if uso_dual else "medio" if contrato_internacional else "bajo"}

def registrar_cliente(caso):
    return {"cliente_id": f"CLI-2026-{caso['id']}", "estado": "pendiente_aprobacion",
            "tipo": caso.get("tipo_cliente","B2B"), "nombre": caso.get("nombre",""),
            "volumen_anual": caso.get("volumen_anual_eur",0)}

def procesar_onboarding_m2(caso):
    info_comercial  = verificar_cliente(caso["id"], caso.get("nombre",""), caso.get("pais","España"), caso.get("tipo_cliente","B2B"))
    info_compliance = verificar_compliance_kyc(caso)
    info_credito    = verificar_credito(caso.get("volumen_anual_eur",0), caso.get("antiguedad_empresa_años",0), caso.get("documentacion_completa",True), caso.get("tipo_cliente","B2B"))
    info_legal      = verificar_legal_contrato(caso.get("pais","España"), caso.get("sector",""), caso.get("volumen_anual_eur",0), caso.get("uso_dual",False))
    registro        = registrar_cliente(caso)
    llm_8b = LLM(model="groq/llama-3.1-8b-instant", temperature=0.0)
    agente = Agent(
        role="Gestor de Onboarding",
        goal="Analizar y decidir el alta de nuevos clientes de ACME Industrial S.A.",
        backstory=f"""Eres el gestor de onboarding de ACME Industrial S.A.
Cliente: {caso['descripcion']}
Info comercial: {json.dumps(info_comercial, ensure_ascii=False)}
Compliance/KYC: {json.dumps(info_compliance, ensure_ascii=False)}
Crédito: {json.dumps(info_credito, ensure_ascii=False)}
Legal: {json.dumps(info_legal, ensure_ascii=False)}
Registro: {json.dumps(registro, ensure_ascii=False)}
Responde SOLO con JSON: {{"decision": "aprobacion_automatica"|"aprobacion_con_condiciones"|"solicitud_documentacion"|"rechazo"|"escalacion_hil", "razon": "texto breve", "escala_hil": true|false}}
REGLA: escala_hil=true SOLO si hay alerta AML, PEP, UBO sancionado, uso dual, o conflicto de interés.""",
        llm=llm_8b, verbose=False)
    tarea = Task(description=f"Decide el alta del cliente: {caso['nombre']} — {caso['descripcion'][:200]}", agent=agente, expected_output='JSON con decision, razon, escala_hil')
    crew = Crew(agents=[agente], tasks=[tarea], verbose=False)
    try:
        t_ini = time.time()
        resultado = crew.kickoff()
        t_fin = time.time()
        texto = str(resultado)
        decision = "error"; escala = False
        try:
            match = re.search(r'\{[^{}]*"decision"[^{}]*\}', texto, re.DOTALL)
            if match:
                datos = json.loads(match.group())
                decision = datos.get("decision","error")
                escala = datos.get("escala_hil",False)
        except: pass
        if decision == "error":
            tl = texto.lower()
            if "escala" in tl or "hil" in tl: decision="escalacion_hil"; escala=True
            elif "rechazo" in tl: decision="rechazo"
            elif "condicion" in tl or "document" in tl: decision="aprobacion_con_condiciones"
            else: decision="aprobacion_automatica"
        return {"caso_id": caso["id"], "nivel": caso["nivel"], "decision": decision, "escalo_hil": escala, "tiempo": round(t_fin-t_ini,2), "ground_truth": caso["ground_truth"]}
    except Exception as e:
        return {"caso_id": caso["id"], "nivel": caso["nivel"], "decision": "error", "escalo_hil": False, "tiempo": 0, "ground_truth": caso["ground_truth"], "error": str(e)[:200]}

# EJECUCIÓN M2
os.makedirs("resultados_onboarding", exist_ok=True)
print("Ejecutando M2 — Single Agent (20 casos)...\n")
resultados_onb_m2 = []
for i, caso in enumerate(casos_onb):
    print(f"  {caso['id']}...", end=" ", flush=True)
    for intento in range(3):
        r = procesar_onboarding_m2(caso)
        if r["decision"] != "error": break
        espera = 60*(intento+1)
        print(f"\n    Reintento {intento+1} — esperando {espera}s...", end=" ")
        time.sleep(espera)
    resultados_onb_m2.append(r)
    print(f"{r['decision']} ({'HiL' if r['escalo_hil'] else 'auto'})")
    time.sleep(60)

ARR,HDA,DER,PT = calcular_metricas(resultados_onb_m2, 20)
print(f"\n{'='*40}\nM2 — ARR={ARR}% | HDA={HDA}% | DER={DER}% | PT={PT}s\n{'='*40}")
with open("resultados_onboarding/resultados_onb_m2.json","w",encoding="utf-8") as f:
    json.dump({"modelo":"M2_single_onboarding","metricas":{"ARR":ARR,"HDA":HDA,"DER":DER,"PT":PT},"resultados_por_caso":resultados_onb_m2}, f, ensure_ascii=False, indent=2)
print("✅ Guardado resultados_onb_m2.json")

Ejecutando M2 — Single Agent (20 casos)...

  C01... aprobacion_automatica (auto)
  C02... aprobacion_automatica (auto)
  C03... aprobacion_automatica (auto)
  C04... aprobacion_automatica (auto)
  C05... aprobacion_automatica (auto)
  C06... aprobacion_automatica (auto)
  C07... aprobacion_automatica (auto)
  C08... aprobacion_automatica (auto)
  C09... aprobacion_automatica (auto)
  C10... solicitud_documentacion (auto)
  C11... rechazo (auto)
  C12... aprobacion_con_condiciones (auto)
  C13... aprobacion_automatica (auto)
  C14... rechazo (auto)
  C15... aprobacion_con_condiciones (HiL)
  C16... rechazo (HiL)
  C17... rechazo (auto)
  C18... rechazo (HiL)
  C19... aprobacion_con_condiciones (HiL)
  C20... rechazo (HiL)

M2 — ARR=75.0% | HDA=83.3% | DER=45.0% | PT=0.6s
✅ Guardado resultados_onb_m2.json


M2 supera el objetivo ARR>70% y detecta 5 de 6 casos HiL (HDA=83.3%). El caso no detectado (C17, PEP) requiere combinar el flag PEP con el volumen inusualmente alto — señal transversal que el agente único no siempre integra. DER=45% refleja que el espacio de decisión en onboarding es más granular que en procurement.

## M4 — Multi-Agent Mixto (8b×3 + 70b×1)

In [10]:
def procesar_onboarding_m4(caso):
    """Multi-Agent mixto: 8b x3 + 70b x1 (Compliance decisor)"""
    import time, re, json
    from crewai import Agent, Task, Crew, LLM

    MODELO_8B  = "groq/llama-3.1-8b-instant"
    MODELO_70B = "groq/llama-3.3-70b-versatile"
    llm_8b  = LLM(model=MODELO_8B,  temperature=0.0)
    llm_70b = LLM(model=MODELO_70B, temperature=0.0)

    info_comercial  = verificar_cliente(caso["id"], caso.get("nombre",""),
                                        caso.get("pais","España"),
                                        caso.get("tipo_cliente","B2B"))
    info_compliance = verificar_compliance_kyc(caso)
    info_credito    = verificar_credito(
                        caso.get("volumen_anual_eur", 0),
                        caso.get("antiguedad_empresa_años", 0),
                        caso.get("documentacion_completa", True),
                        caso.get("tipo_cliente", "B2B"))
    info_legal      = verificar_legal_contrato(
                        caso.get("pais","España"),
                        caso.get("sector",""),
                        caso.get("volumen_anual_eur", 0),
                        caso.get("uso_dual", False))
    registro        = registrar_cliente(caso)

    comercial = Agent(
        role="Agente Comercial",
        goal="Validar el perfil comercial del cliente y su potencial.",
        backstory=f"Cliente: {caso['descripcion']}\nInfo comercial: {json.dumps(info_comercial, ensure_ascii=False)}",
        llm=llm_8b, verbose=False
    )
    legal = Agent(
        role="Asesor Legal",
        goal="Revisar requisitos legales y contractuales del alta.",
        backstory=f"Info legal: {json.dumps(info_legal, ensure_ascii=False)}",
        llm=llm_8b, verbose=False
    )
    finanzas = Agent(
        role="Analista Financiero",
        goal="Evaluar el riesgo crediticio y establecer límite de crédito.",
        backstory=f"Info crédito: {json.dumps(info_credito, ensure_ascii=False)}",
        llm=llm_8b, verbose=False
    )
    compliance = Agent(
        role="Auditor de Compliance (KYC/AML)",
        goal="Tomar la decisión final de onboarding basándose en todos los factores de riesgo.",
        backstory=f"""Eres el decisor final de onboarding. Tienes toda la información:
Cliente: {caso['descripcion']}
Compliance/KYC: {json.dumps(info_compliance, ensure_ascii=False)}
Crédito: {json.dumps(info_credito, ensure_ascii=False)}
Legal: {json.dumps(info_legal, ensure_ascii=False)}
Comercial: {json.dumps(info_comercial, ensure_ascii=False)}

Responde SOLO con JSON:
{{"decision": "aprobacion_automatica"|"aprobacion_con_condiciones"|"solicitud_documentacion"|"rechazo"|"escalacion_hil",
  "razon": "texto breve",
  "escala_hil": true|false}}

REGLA CRÍTICA: escala_hil=true OBLIGATORIO si:
- alerta_aml=true, alerta_ubo_sancionado=true, pep_detectado=true
- uso_dual=true, conflicto_interes=true, alerta_pais_riesgo=true""",
        llm=llm_70b, verbose=False
    )

    t1 = Task(description=f"Valida perfil comercial de {caso['nombre']}", agent=comercial,
              expected_output="Evaluación comercial")
    t2 = Task(description="Revisa requisitos legales y contractuales", agent=legal,
              expected_output="Evaluación legal")
    t3 = Task(description="Evalúa riesgo crediticio y límite", agent=finanzas,
              expected_output="Evaluación financiera")
    t4 = Task(description="Decide en JSON el alta del cliente", agent=compliance,
              expected_output="JSON con decision, razon, escala_hil")

    crew = Crew(agents=[comercial, legal, finanzas, compliance],
                tasks=[t1, t2, t3, t4], verbose=False)

    try:
        t_ini = time.time()
        resultado = crew.kickoff()
        t_fin = time.time()
        texto = str(resultado)

        decision = "error"; escala = False
        try:
            match = re.search(r'\{[^{}]*"decision"[^{}]*\}', texto, re.DOTALL)
            if match:
                datos = json.loads(match.group())
                decision = datos.get("decision", "error")
                escala   = datos.get("escala_hil", False)
        except:
            pass
        if decision == "error":
            tl = texto.lower()
            if "escala" in tl or "hil" in tl:
                decision = "escalacion_hil"; escala = True
            elif "rechazo" in tl or "rechaz" in tl:
                decision = "rechazo"
            elif "condicion" in tl or "document" in tl:
                decision = "aprobacion_con_condiciones"
            else:
                decision = "aprobacion_automatica"

        return {
            "caso_id": caso["id"], "nivel": caso["nivel"],
            "decision": decision, "escalo_hil": escala,
            "tiempo": round(t_fin - t_ini, 2),
            "ground_truth": caso["ground_truth"]
        }
    except Exception as e:
        return {
            "caso_id": caso["id"], "nivel": caso["nivel"],
            "decision": "error", "escalo_hil": False,
            "tiempo": 0, "ground_truth": caso["ground_truth"],
            "error": str(e)[:200]
        }

In [11]:
print("Ejecutando M4 — Multi-Agent mixto (20 casos)...\n")
resultados_onb_m4 = []
for i, caso in enumerate(casos_onb):
    print(f"  {caso['id']}...", end=" ", flush=True)
    for intento in range(3):
        r = procesar_onboarding_m4(caso)
        if r["decision"] != "error":
            break
        espera = 70 * (intento + 1)
        print(f"\n    Reintento {intento+1} — esperando {espera}s...", end=" ")
        time.sleep(espera)
    resultados_onb_m4.append(r)
    print(f"{r['decision']} ({'HiL' if r['escalo_hil'] else 'auto'})")
    time.sleep(70)

ARR,HDA,DER,PT = calcular_metricas(resultados_onb_m4, 20)
print(f"\n{'='*40}")
print(f"M4 — ARR={ARR}% | HDA={HDA}% | DER={DER}% | PT={PT}s")
print(f"{'='*40}")
with open("resultados_onboarding/resultados_onb_m4.json","w",encoding="utf-8") as f:
    json.dump({"modelo":"M4_mixto_onboarding",
               "metricas":{"ARR":ARR,"HDA":HDA,"DER":DER,"PT":PT},
               "resultados_por_caso":resultados_onb_m4}, f, ensure_ascii=False, indent=2)
print("✅ Guardado resultados_onb_m4.json")

Ejecutando M4 — Multi-Agent mixto (20 casos)...

  C01... aprobacion_automatica (auto)
  C02... aprobacion_automatica (auto)
  C03... aprobacion_automatica (auto)
  C04... aprobacion_automatica (auto)
  C05... aprobacion_automatica (auto)
  C06... aprobacion_automatica (auto)
  C07... aprobacion_automatica (auto)
  C08... aprobacion_automatica (auto)
  C09... aprobacion_automatica (auto)
  C10... solicitud_documentacion (auto)
  C11... solicitud_documentacion (auto)
  C12... aprobacion_automatica (auto)
  C13... aprobacion_automatica (auto)
  C14... solicitud_documentacion (auto)
  C15... 
    Reintento 1 — esperando 70s... aprobacion_con_condiciones (HiL)
  C16... escalacion_hil (HiL)
  C17... escalacion_hil (HiL)
  C18... escalacion_hil (HiL)
  C19... escalacion_hil (HiL)
  C20... escalacion_hil (HiL)

M4 — ARR=70.0% | HDA=100.0% | DER=40.0% | PT=4.68s
✅ Guardado resultados_onb_m4.json


M4 detecta los 6 casos HiL perfectamente (HDA=100%) gracias al modelo 70b en el agente decisor. ARR=70% está justo en el objetivo — el sistema es más conservador que M2 porque los agentes especializados bloquean más casos para revisión. DER=40% ligeramente mejor que M2.

## Resumen — Experimento 03

In [12]:
print("="*50)
print("RESUMEN EXPERIMENTO 03 — ONBOARDING DE CLIENTES")
print("="*50)
print(f"{'Modelo':<20} {'ARR':>6} {'HDA':>6} {'DER':>6} {'PT':>8}")
print("-"*50)
for nombre, resultados in [("M1 RPA", resultados_onb_m1),
                             ("M2 Single 8b", resultados_onb_m2),
                             ("M4 Multi mixto", resultados_onb_m4)]:
    a,h,d,p = calcular_metricas(resultados, 20)
    print(f"{nombre:<20} {a:>5}% {h:>5}% {d:>5}% {p:>7}s")
print("="*50)
print("\nResultados guardados en resultados_onboarding/")

RESUMEN EXPERIMENTO 03 — ONBOARDING DE CLIENTES
Modelo                  ARR    HDA    DER       PT
--------------------------------------------------
M1 RPA               100.0%   0.0%  70.0%     0.0s
M2 Single 8b          15.0%     0%   0.0%    0.11s
M4 Multi mixto        70.0% 100.0%  40.0%    4.68s

Resultados guardados en resultados_onboarding/


El hallazgo clave es que las señales de riesgo en onboarding (PEP, AML, UBO) son más explícitas que en procurement (GDPR, AESA, CE) pero menos que en incident management (flags booleanos). Esto explica que HDA quede entre los dos experimentos anteriores: 30.8% → 83.3-100% → 100%.

In [17]:
!git push --set-upstream origin main

branch 'main' set up to track 'origin/main'.


To https://github.com/Edurne0781/multiagent-procurement-llm.git
   c15b511..13141c9  main -> main
